# P1 · Frame teacher F
Run each cell in order. Enable Internet in Kaggle settings; select a GPU for training. Read docs/p1/P1_INSTRUCTION.md first.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)], check=True)
os.chdir(project)
subprocess.run(["git","pull","--ff-only","origin","main"],check=True)
subprocess.run([sys.executable,"-m","pip","install","--no-deps","-e","."],check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy
print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve
raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not cache.exists():
    subprocess.run([sys.executable, '-m', 'acr.data.billboard.prepare'], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
# First training run. Select a NEW output name for every retry.
overfit_run = project / "runs/p1-overfit-kaggle"
subprocess.run([sys.executable, '-m', 'acr.training.frame', "--config", "configs/p1/p1-overfit.json",
                "--output", str(overfit_run), "--device", device], check=True)
metrics = json.loads((overfit_run / "metrics.json").read_text())
print(metrics)
assert metrics["overfit_pass"], "Inspect history and data before running the full baseline."


## Full baseline
After the overfit check passes, execute the next cell explicitly. It trains on 367 labeled songs and evaluates on 110 validation songs. Save the executed notebook version and download the run archive.

In [ ]:
RUN_BASELINE = False  # Set True only after reviewing the overfit result.
baseline_run = project / "runs/p1-baseline-kaggle"
if RUN_BASELINE:
    assert json.loads((overfit_run / "metrics.json").read_text())["overfit_pass"]
    subprocess.run([sys.executable, '-m', 'acr.training.frame', "--config", "configs/p1/p1-baseline.json",
                    "--output", str(baseline_run), "--device", device], check=True)
    print((baseline_run / "metrics.json").read_text())


In [ ]:
# Inspect one validation timeline before downloading the run.
import matplotlib.pyplot as plt
prediction_file = next(((baseline_run if RUN_BASELINE else overfit_run) / "predictions").glob("*.npz"))
with numpy.load(prediction_file) as data:
    plt.figure(figsize=(14, 4))
    for name in ("reference", "prediction"):
        values = data[name].astype(float)
        values[values < 0] = numpy.nan
        plt.step(data["times"], values, where="post", label=name, alpha=.8)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Chord class ID (see run.json)")
    plt.legend()
    plt.tight_layout()
    plt.show()
import shutil
archive = shutil.make_archive("/kaggle/working/p1-results", "zip", project, "runs")
print(archive)
